# Submission notebook — runs with internet OFFThis is a **code competition**: `test.csv` in the public copy holds 3placeholder studies, so the real test set is mounted only when Kaggle re-runsthis notebook. A CSV cannot be submitted directly; this notebook is thesubmission. See `SUBMITTING.md`.It downloads nothing. Point `MODEL` at a self-contained checkpoint holdingboth state dicts (step A of SUBMITTING.md), or at a full`ConvNeXtFrozenPool` checkpoint — both layouts are handled below.Run order: Save Version → Save & Run All (Commit), with GPU on and internetoff, then submit the finished version.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
MODEL = "/kaggle/input/knee-selfcontained/knee_model_selfcontained.pt"
OUT = "/kaggle/working/submission.csv"
SLICE_BATCH = 32

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| internet must be OFF for this to be a valid commit")

TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Model — no weight downloadTwo accepted layouts. A self-contained file has `backbone_state_dict` plus`head_state_dict`. A full `ConvNeXtFrozenPool` checkpoint has`model_state_dict` with `backbone.` and `classifier.` prefixes. Anything thatwould reach the network is a bug here, so the architecture is built with`weights=None` and every tensor comes from the file.

In [ ]:
ck = torch.load(MODEL, map_location="cpu", weights_only=False)

name = ck.get("backbone", "convnext_small")
dim = ck.get("feature_dim", {"convnext_small": 768, "convnext_base": 1024,
                             "resnet50": 2048}[name])
PLANE = ck.get("plane", "axial")

ctor = {"convnext_small": models.convnext_small,
        "convnext_base": models.convnext_base,
        "resnet50": models.resnet50}[name]
backbone = ctor(weights=None)                       # weights=None: no download
if name.startswith("convnext"):
    backbone.classifier = nn.Identity()
else:
    backbone.fc = nn.Identity()

head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, 256), nn.GELU(),
                     nn.Dropout(0.3), nn.Linear(256, 12))

if "backbone_state_dict" in ck:
    backbone.load_state_dict(ck["backbone_state_dict"], strict=True)
    head.load_state_dict(ck["head_state_dict"], strict=True)
elif "model_state_dict" in ck:
    sd = ck["model_state_dict"]
    backbone.load_state_dict(
        {k[len("backbone."):]: v for k, v in sd.items()
         if k.startswith("backbone.")}, strict=True)
    head.load_state_dict(
        {k[len("classifier."):]: v for k, v in sd.items()
         if k.startswith("classifier.")}, strict=True)
else:
    raise KeyError("checkpoint has neither backbone_state_dict nor "
                   "model_state_dict; see SUBMITTING.md step A")

backbone = backbone.eval().to(DEVICE)
head = head.eval().to(DEVICE)
print(f"loaded {name} ({dim}-d), plane={PLANE}, "
      f"cv={ck.get('cv_macro_auc_56_studies')}")

## Series selection, with a fallbackAxial fluid-sensitive covers every training study; sagittal misses 257 of4,407. A study with no series for the head's own plane falls back to the nextavailable plane, which is a degraded guess but far better than the 0.5 row itwould otherwise get. The count of each is printed, so a surprise shows up inthe log rather than silently in the score.

In [ ]:
sub_template = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGETS = [c for c in sub_template.columns if c != "StudyInstanceUID"]
assert len(TARGETS) == 12, TARGETS

test_series = pd.read_csv(os.path.join(DATA_DIR, "test_series.csv"))
studies = sub_template["StudyInstanceUID"].tolist()
print(f"{len(studies)} test studies, {len(test_series)} test series")

order = [PLANE] + [p for p in ("axial", "coronal", "sagittal") if p != PLANE]
chosen, used = {}, {}
for plane in order:
    sel = test_series[
        (test_series["Anatomical_Plane"].str.lower() == plane)
        & (test_series["Fluid_Sensitive"] == 1)
        & (test_series["Fat_Suppression"] == 1)]
    one = (sel.sort_values(["StudyInstanceUID", "SeriesInstanceUID"])
              .drop_duplicates("StudyInstanceUID", keep="first"))
    for _, r in one.iterrows():
        sid = r["StudyInstanceUID"]
        if sid not in chosen:
            chosen[sid] = r["SeriesInstanceUID"]
            used[sid] = plane

native = sum(1 for s in studies if used.get(s) == PLANE)
fallback = sum(1 for s in studies if s in chosen) - native
none_at_all = sum(1 for s in studies if s not in chosen)
print(f"  on {PLANE}: {native} | on a fallback plane: {fallback} | "
      f"no usable series: {none_at_all}")

## Slice reading — DICOM, with JPEG tolerated

In [ ]:
def read_slices(d):
    files = sorted(os.listdir(d))
    if files and files[0].lower().endswith((".jpg", ".jpeg", ".png")):
        return [TF(Image.open(os.path.join(d, f)).convert("RGB")) for f in files]

    import pydicom
    ds = []
    for f in files:
        p = os.path.join(d, f)
        if not os.path.isfile(p):
            continue
        try:
            x = pydicom.dcmread(p)
        except Exception:
            continue
        if hasattr(x, "PixelData"):
            ds.append(x)
    if not ds:
        raise RuntimeError(f"no readable slices in {d}")

    if hasattr(ds[0], "ImageOrientationPatient") and \
       hasattr(ds[0], "ImagePositionPatient"):
        iop = np.asarray(ds[0].ImageOrientationPatient, dtype=np.float32)
        nrm = np.cross(iop[:3], iop[3:])
        ds = [ds[i] for i in np.argsort([
            float(np.dot(np.asarray(x.ImagePositionPatient, dtype=np.float32), nrm))
            for x in ds])]
    elif all(hasattr(x, "InstanceNumber") for x in ds):
        ds = sorted(ds, key=lambda x: int(x.InstanceNumber))

    out = []
    for x in ds:
        a = x.pixel_array.astype(np.float32)
        lo, hi = float(a.min()), float(a.max())
        a = (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)
        out.append(TF(Image.fromarray((a * 255).astype(np.uint8)).convert("RGB")))
    return out

## PredictMean-pool the slice features, exactly as in training, then the head. A studythat cannot be read at all gets 0.5 across the board rather than beingdropped — a missing row fails the submission outright.

In [ ]:
rows, failed = [], 0
for i, sid in enumerate(studies):
    if sid not in chosen:
        rows.append([sid] + [0.5] * 12)
        continue
    d = os.path.join(DATA_DIR, "test_series", sid, chosen[sid])
    try:
        xs = torch.stack(read_slices(d)).to(DEVICE)
        with torch.no_grad():
            feats = torch.cat([
                backbone(xs[j:j + SLICE_BATCH]).reshape(
                    min(SLICE_BATCH, len(xs) - j), -1)
                for j in range(0, len(xs), SLICE_BATCH)])
            p = torch.sigmoid(head(feats.mean(0, keepdim=True)))[0]
        rows.append([sid] + [float(v) for v in p.float().cpu().numpy()])
    except Exception as exc:
        failed += 1
        if failed <= 5:
            print(f"  {sid}: {exc} -> 0.5")
        rows.append([sid] + [0.5] * 12)
    if (i + 1) % 200 == 0:
        print(f"  {i + 1}/{len(studies)}", flush=True)

print(f"done, {failed} studies fell back to 0.5")

## Write and check before the commit ends

In [ ]:
sub = pd.DataFrame(rows, columns=["StudyInstanceUID"] + TARGETS)

# column order and row set must match the template exactly
sub = sub[sub_template.columns]
assert list(sub.columns) == list(sub_template.columns)
assert len(sub) == len(sub_template), (len(sub), len(sub_template))
assert set(sub["StudyInstanceUID"]) == set(sub_template["StudyInstanceUID"])
assert sub["StudyInstanceUID"].nunique() == len(sub)
assert np.isfinite(sub[TARGETS].to_numpy()).all(), "non-finite predictions"
assert sub[TARGETS].to_numpy().min() >= 0 and sub[TARGETS].to_numpy().max() <= 1

sub.to_csv(OUT, index=False)
print(f"wrote {OUT}  shape {sub.shape}")
print(sub.head(3).to_string(index=False))
print("\nper-target mean prediction:")
print(sub[TARGETS].mean().round(3).to_string())